In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/step00_IMPORTS

In [0]:
def null_or_blank(df):
  null_or_blank = df.select(
                          [
                            F.count(
                                     F.when(
                                             F.col(c).isNull()
                                             | (F.col(c).cast('string') == '') 
                                             | (F.col(c).cast('string').rlike('^\\s*$')),
                                             c
                                           )
                                   ).alias(c)
                            for c in df.columns
                          ]
                         )
  return null_or_blank

In [0]:
def check_duplicates(df, column):
  columns = column if isinstance(column, (list, tuple)) else [column]
  return df.groupBy(*columns).count().filter(F.col('count') > 1).show()

In [0]:
def check_and_remove_duplicates(df, column):
    columns = column if isinstance(column, (list, tuple)) else [column]
    check_duplicates(df, columns)
    if df.groupBy(*columns).count().filter(F.col('count') > 1).count() > 0:
        print('Duplicates removed.')
        return df.dropDuplicates(columns)
    print('No duplicates found.')
    return df

In [0]:
def check_valid_federative_unit(df, column):
  valid_ufs = ['AC', 'AL', 'AP', 'AM', 'BA', 'CE', 'DF', 'ES', 'GO', 'MA', 'MT',
               'MS', 'MG', 'PA', 'PB', 'PR', 'PE', 'PI', 'RJ', 'RN', 'RS',
               'RO', 'RR', 'SC', 'SP', 'SE', 'TO']
  invalid = df.filter(~F.upper(F.col(column)).isin(valid_ufs))
  invalid_count = invalid.count()
  print(f'Invalid federative unit values in "{column}": {invalid_count}')
  if invalid_count > 0:
    invalid.select(column, F.count(F.lit(1)).over()) if False else None
    invalid.groupBy(column).count().orderBy(F.desc('count')).show()
  else:
    print('All values are valid federative units.')
  return invalid